# `vkpanel` Demo — HoloViz Panel-Style Reactive GPU Dashboards

Demonstrates `vkpanel` widgets (`IntSlider`, `FloatSlider`, `Select`, `Checkbox`, `DataTable`), reactive `pn.bind(...)`, and layouts (`Row`, `Column`, `Card`, `Tabs`, `Markdown`) connected directly to Vulkan GPU `vknumpy` + `vkshader` + `vkholoviews`.

In [1]:
import sys, time
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vknumpy").is_dir()), Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import vknumpy as vnp
import vkpandas as vpd
import vkshader as vks
import vkholoviews as hv
import vkpanel as pn

# Pre-generate a 2M-point multi-cluster GPU dataset
rng = np.random.default_rng(2026)
n = 2_000_000
x_g = vnp.array(rng.normal(0.0, 1.0, size=n).astype(np.float32))
y_g = vnp.array(rng.normal(0.0, 1.0, size=n).astype(np.float32))
r_g = vnp.sqrt(x_g * x_g + y_g * y_g)
df = vpd.DataFrame({"x": x_g, "y": y_g, "radius": r_g})
sp = vks.SpatialPoints(x_g, y_g)

# Reactive widgets
how_dd   = pn.Select(name="Shading", options=["eq_hist", "log", "cbrt", "linear"], value="eq_hist")
zoom_sl  = pn.FloatSlider(name="Extent", start=1.0, end=5.0, step=0.25, value=3.5)
bins_sl  = pn.IntSlider(name="Hist Bins", start=15, end=80, step=5, value=40)

def _render_density(how, extent):
    t0 = time.perf_counter()
    cvs = vks.Canvas(460, 420, x_range=(-extent, extent), y_range=(-extent, extent))
    agg = cvs.points(sp)
    img = vks.shade(agg, how=how, basemap=None, spread_px=0)
    return img

def _render_hist(bins, extent):
    return hv.Histogram(df, "radius", bins=bins, range=(0.0, extent)).opts(
        width=460, height=300, title=f"GPU Radius Histogram ({bins} bins, 2M pts)"
    )

summary_df = vpd.DataFrame({
    "Metric": ["Points", "Mean Radius", "Std Radius", "Max Radius"],
    "Value": [
        float(n),
        round(float(vnp.mean(r_g).item()), 4),
        round(float(vnp.std(r_g).item()), 4),
        round(float(vnp.max(r_g).item()), 4),
    ],
})

tab_density = pn.Column(
    pn.Row(how_dd, zoom_sl),
    pn.bind(_render_density, how_dd, zoom_sl),
)
tab_hist = pn.Column(
    pn.Row(bins_sl, zoom_sl),
    pn.bind(_render_hist, bins_sl, zoom_sl),
)
tab_stats = pn.Card(pn.DataTable(summary_df, max_rows=10), title="GPU Summary Statistics")

pn.Column(
    pn.Markdown("### Live Vulkan GPU Dashboard (2,000,000 points)"),
    pn.Tabs(('Density Shade', tab_density), ('Radius Histogram', tab_hist), ('Stats Table', tab_stats)),
)

ggml_vulkan: Found 1 Vulkan devices:
ggml_vulkan: 0 = AMD Custom GPU 0405 (RADV VANGOGH) (radv) | uma: 1 | rebar: 1 | fp16: 0 | bf16: 0 | fp4: 0 | warp size: 32 | shared memory: 65536 | int dot: 1 | matrix cores: none


[0 AMD Custom GPU 0405 (RADV VANGOGH)]  queueC=1[4]  queueT=0[1]  prefer-slm=0  rebar=1  r-score=11
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  fp16-p/s/u/a=1/1/1/1  int8-p/s/u/a=1/1/1/1  bf16-p/s=1/0
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  subgroup=64(32~64)  ops=1/1/1/1/1/1/1/1/1/1
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  fp16-cm=0  int8-cm=0  bf16-cm=0  fp8-cm=0  fp6-cm=0  fp4-cm=0
